# Module 04.12 — Connectors (`action`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.12 Connectors (`action`)

Connectors (saved as type `action`) define the **delivery channel** for alerting actions:
Slack, email, PagerDuty, Jira, webhook, server log, and many others. A connector stores
connection details (URL, channel name, API endpoint) and credentials (API key, password,
OAuth token). Rules reference connectors via their `references` array; one connector can
be shared by many rules.

The critical thing to understand about connectors from a restore perspective is
**secret encryption**. Kibana encrypts the `secrets` field (passwords, API keys) at rest
using the `xpack.encryptedSavedObjects.encryptionKey` value set in `kibana.yml`. The
ciphertext is stored in the `.kibana` index and is included in a feature-state snapshot.

This creates an important cross-cluster restore constraint:
> If you restore a `kibana` feature state to a cluster whose Kibana uses a **different**
> `encryptedSavedObjects.encryptionKey`, the secrets will be unreadable. The connector
> objects will exist, but any rule that tries to fire an action will fail with a
> decryption error. You must re-enter the credentials for each affected connector.

To avoid this, ensure all target clusters use the same encryption key, or document
which connectors need their secrets re-entered as part of your DR runbook.

In [ ]:
heading('4.12 Connector — create server log connector')

connector_resp = kibana_post('/api/actions/connector', {
    'name': 'Training Server Log',
    'connector_type_id': '.server-log',
    'config': {},
    'secrets': {},
})
connector_id = connector_resp['id']
success(f'Connector created: id={connector_id}')

# Show the saved object
connectors_so = find_saved_objects('action')
training_connector = next((c for c in connectors_so if c['id'] == connector_id), None)
if training_connector:
    pp(training_connector, 'action (connector) saved object')
    info('Key fields:')
    info('  actionTypeId    — connector type identifier (.server-log, .slack, .email, etc.)')
    info('  config          — non-secret config (e.g. webhook URL, channel name)')
    info('  secrets         — encrypted credentials (not exposed in GET responses)')
    info('  NOTE: secrets are encrypted with the Kibana encryption key.')
    info('  If you restore to a cluster with a DIFFERENT encryption key, connectors with')
    info('  secrets will need their credentials re-entered after restore.')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link(
    '/app/management/insightsAndAlerting/triggersActions/connectors',
    'Stack Management → Connectors — view the server log connector',
)

In [ ]:
# Update the rule to reference the connector
heading('Link connector to rule')

kibana_post(f'/api/alerting/rule/{rule_id}', {
    'name': 'Training — High Order Volume',
    'schedule': {'interval': '1h'},
    'params': {
        'index': ['kibana_sample_data_ecommerce'],
        'timeField': 'order_date',
        'esQuery': json.dumps({'query': {'match_all': {}}}),
        'size': 100,
        'thresholdComparator': '>',
        'threshold': [500],
        'timeWindowSize': 24,
        'timeWindowUnit': 'h',
    },
    'actions': [{
        'id': connector_id,
        'group': 'query matched',
        'params': {'message': 'High order volume detected!', 'level': 'warn'},
    }],
    'notify_when': 'onActionGroupChange',
})

# Verify the reference
updated_rule_so = kibana_get(f'/api/saved_objects/alert/{rule_id}')
action_refs = [r for r in updated_rule_so.get('references', []) if r['type'] == 'action']
success(f'Rule now references connector(s): {action_refs}')

In [ ]:
heading('Connector — snapshot → delete → restore')

# Ensure connector exists (re-runnable)
existing_connectors = find_saved_objects('action')
if not any(o['id'] == connector_id for o in existing_connectors):
    warn('Connector missing — recreating')
    conn_resp = kibana_post('/api/actions/connector', {
        'name': 'Training — Server Log',
        'connector_type_id': '.server-log',
        'config': {}, 'secrets': {},
    })
    connector_id = conn_resp['id']
    success(f'Connector recreated: id={connector_id}')

snap_delete_restore_cycle('snap-connector-test', 'Connector')

kibana_delete(f'/api/actions/connector/{connector_id}')
warn(f'Accidentally deleted connector {connector_id}')
assert not any(o['id'] == connector_id for o in find_saved_objects('action')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-connector-test')
time.sleep(3)

restored = find_saved_objects('action')
assert any(o['id'] == connector_id for o in restored), 'Connector should be restored'
success(f'Connector {connector_id} successfully restored!')
kibana_link('/app/management/insightsAndAlerting/triggersActions/connectors', 'Connectors — verify restored connector')